# KOSIS ChromaDB 하이브리드 좌표 검색 → 2차 READY (Colab GPU)

1차 READY 이후의 ITEM/OBJ 좌표 후보를 ChromaDB dense + lexical + reranker 로 뽑고,
기존 KOSIS API 검증(`kosis_validate_mapping_candidates.py`)에 그대로 연결한다.

**원칙**: 임베딩/리랭커 점수는 후보 생성·순위에만 쓴다. READY 는 공식 메타 + KOSIS API 결과로만 확정한다.

**셀 순서**: 1 GPU 확인 → 2 저장소·의존성 → 3 Drive 마운트 → 4 입력 확인 →
5 Chroma 인덱스 생성 → 6 하이브리드 검색 → 7 API 검증 → 8 실제값 검증 →
9 A/B/C 평가 → 10 Drive 저장

In [ ]:
# 1. GPU 확인 (런타임 → 런타임 유형 변경 → GPU)
!nvidia-smi -L

In [ ]:
# 2. 저장소 + 의존성
!git clone https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git repo
%cd repo
!git checkout codex/repro-baseline-20260727
!pip install -q -r requirements-ml.txt

In [ ]:
# 3. Drive 마운트 (기존 07_mapping_jinsung 결과 재사용)
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''  # 코드/CSV/로그에 키를 남기지 않는다
if not os.environ['KOSIS_API_KEY']:
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')

RUN = '/content/drive/MyDrive/NLP_05-Team-Project-3/runs/contextual_top50_context_v2_8x3/07_mapping_jinsung'
OUT = RUN + '/chroma_hybrid'
!mkdir -p {OUT}
!ls {RUN}

In [ ]:
# 4. 입력 확인 (평가 대상 measurement 를 여기서 고정한다)
import pandas as pd
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
meta = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_meta_index.csv')
cand = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_table_candidates.csv')
print('1차 READY measurement:', ready['claim_measurement_id'].nunique())
print('meta rows:', len(meta), '| table candidates:', len(cand))

In [ ]:
# 5. Chroma 좌표 인덱스 생성 (BGE-M3 임베딩을 직접 저장 → manifest 로 모델·차원 고정)
!python kosis_build_chroma_meta_index.py \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --embedding-model BAAI/bge-m3 \
  --prd-se-source {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --device cuda \
  --reset
!cat data/indexes/kosis_meta_chroma/chroma_manifest.json

In [ ]:
# 6. 하이브리드 검색 (metadata filter → dense → lexical → RRF → reranker → Top-10)
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --stats-output {OUT}/chroma_search_stats.csv \
  --dense-top-k 50 --lexical-top-k 50 --rerank-top-k 20 --final-top-k 10 \
  --reranker-model BAAI/bge-reranker-v2-m3 --device cuda

### 6-1. (선택) 실험 B — Chroma dense 만
`--no-reranker --lexical-top-k 0` 으로 dense 단독 후보를 만들어 A/B/C 비교에 쓴다.

In [ ]:
!python kosis_chroma_hybrid_search.py \
  --claims {RUN}/05_hcx_measurements_kosis_ready.csv \
  --table-candidates {RUN}/05_hcx_measurements_kosis_table_candidates.csv \
  --persist-dir data/indexes/kosis_meta_chroma \
  --collection kosis_meta_coordinates \
  --output {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv \
  --dense-top-k 50 --lexical-top-k 0 --rerank-top-k 20 --final-top-k 10 \
  --no-reranker --device cuda

In [ ]:
# 7. 기존 KOSIS API 검증에 연결 (READY만 자동 확정, PROVISIONAL은 수동 검토)
!python kosis_validate_mapping_candidates.py \
  --input {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --meta-index {RUN}/05_hcx_measurements_kosis_meta_index.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --evaluate-all-ranks \
  --strict-seeded-coordinate \
  --item-top-k 1 --obj-top-k 1 --max-combinations 1 \
  --allow-provisional

In [ ]:
# 7-1. measurement 단위 진단 (885 후보행 → measurement 단위로 축약)
!python diagnose_validated_mappings.py \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --output {OUT}/diagnosis_chroma.csv

In [ ]:
# 8. 실제값 검증 (기사일 컬럼이 있어야 REVISION_VINTAGE_RISK 정책이 동작)
import pandas as pd
validated = pd.read_csv(f'{OUT}/05_hcx_measurements_kosis_chroma_validated.csv')
ready = pd.read_csv(f'{RUN}/05_hcx_measurements_kosis_ready.csv')
if 'date' not in validated.columns and 'date' in ready.columns:
    validated = validated.merge(ready[['claim_measurement_id', 'date']],
                                on='claim_measurement_id', how='left')
validated[validated['mapping_status'] == 'READY'].to_csv(
    f'{OUT}/verify_input_chroma.csv', index=False, encoding='utf-8-sig')
print('verify 대상:', (validated['mapping_status'] == 'READY').sum())

In [ ]:
!python kosis_verify_claim_values.py \
  --input {OUT}/verify_input_chroma.csv \
  --output {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --delay 0.12

In [ ]:
# 9. A/B/C 동일 표본 평가 (골드 좌표가 없으면 recall 은 gold_required 로 표시된다)
GOLD = ''  # 예: f'{RUN}/gold_coordinates.csv'
gold_arg = f'--gold {GOLD}' if GOLD else ''

!python evaluate_chroma_hybrid_mapping.py --label A_baseline \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {RUN}/05_hcx_measurements_kosis_candidates_with_meta.csv \
  --validated {RUN}/05_hcx_measurements_kosis_validated_mappings.csv \
  --verified {RUN}/05_hcx_measurements_kosis_verified.csv {gold_arg} \
  --output {OUT}/eval_A.json

!python evaluate_chroma_hybrid_mapping.py --label B_chroma_dense \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_dense_only_candidates.csv {gold_arg} \
  --output {OUT}/eval_B.json

!python evaluate_chroma_hybrid_mapping.py --label C_chroma_hybrid \
  --measurements {RUN}/05_hcx_measurements_kosis_ready.csv \
  --candidates {OUT}/05_hcx_measurements_kosis_chroma_candidates.csv \
  --validated {OUT}/05_hcx_measurements_kosis_chroma_validated.csv \
  --verified {OUT}/05_hcx_measurements_kosis_chroma_verified.csv \
  --stats {OUT}/chroma_search_stats.csv {gold_arg} \
  --output {OUT}/eval_C.json

In [ ]:
# 10. 결과 저장 (Chroma 인덱스는 용량이 크므로 Git 에 커밋하지 않는다)
!cp -r data/indexes/kosis_meta_chroma {OUT}/kosis_meta_chroma_index
!ls -la {OUT}

## 해석 주의
- 후보행 수(measurement × Top-K)를 measurement 실패 건수로 읽지 말 것.
- 골드 좌표(`gold_tbl_id` / `gold_itm_id` / `gold_obj_l1`)가 없으면 recall 은 계산하지 않는다.
- READY 증가만으로 개선을 주장하지 말고, 동일 표본 A/B/C 결과와 수동 검수 Precision 으로 판단한다.